# EEVR Preprocessing and Alignment


## Environment setup


In [1]:
import os
os.chdir("/notebooks")

In [2]:
from lib.install import prepare_env
prepare_env()


Installed versions:
tensorflow: 2.18.0
tensorflow-probability: 0.25.0
tf-keras: 2.18.0
transformers: 4.46.3
protobuf: 4.25.8
datasets: 5.0.0
evaluate: 0.4.6
scikit-learn: 1.9.0
imbalanced-learn: 0.14.2
wordcloud: 1.9.6
umap-learn: 0.5.12

Environment packages OK.


2026-06-21 15:04:52.111578: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1782054292.220880     405 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1782054292.254223     405 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-06-21 15:04:52.611478: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.



TensorFlow check:
TensorFlow: 2.18.0
Built with CUDA: True
GPUs: []


2026-06-21 15:05:00.592932: E external/local_xla/xla/stream_executor/cuda/cuda_driver.cc:152] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: CUDA_ERROR_NO_DEVICE: no CUDA-capable device is detected


## Imports


In [3]:
import pandas as pd
from sklearn.preprocessing import StandardScaler
from pathlib import Path

import string
import subprocess
import sys

import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize

# Download required NLTK resources
for package in ["punkt", "punkt_tab", "stopwords", "wordnet", "omw-1.4"]:
    nltk.download(package, quiet=True)


In [4]:
import numpy as np
import pandas as pd
from pathlib import Path
from IPython.display import display

# ============================================================
# Read data
# ============================================================
text_df = pd.read_csv("EEVR/Data_files/Textdata.csv")
vads_df = pd.read_csv("EEVR/Data_files/VADS.csv")

# remove unnamed columns if they exist
text_df = text_df.drop(columns=[c for c in text_df.columns if str(c).startswith("Unnamed")],errors="ignore")
vads_df = vads_df.drop(columns=[c for c in vads_df.columns if str(c).startswith("Unnamed")],errors="ignore")


# ============================================================
# Convert Video ID to integer
# Handles values like V0, V1, v2, 1, 1.0
# ============================================================
def clean_video_id(df):
    video_id_clean = (
        df["Video ID"]
        .astype(str)
        .str.strip()
        .str.replace(r"^[Vv]", "", regex=True)
    )

    video_id_clean = pd.to_numeric(video_id_clean,errors="coerce")

    if video_id_clean.isna().any():
        print("Rows with invalid Video ID:")
        display(df.loc[video_id_clean.isna(), ["Video ID"]].drop_duplicates())
        raise ValueError("Some Video ID values could not be converted to integer.")

    df["Video ID"] = video_id_clean.astype(int)

    return df


text_df = clean_video_id(text_df)
vads_df = clean_video_id(vads_df)


# ============================================================
# Join text and VADS
# ============================================================
join_cols = [
    "Participant ID",
    "Playlist ID",
    "Video ID",
    "CMA",
    "Arousal",
    "Valence"
]

text_vads_df = text_df.merge(
    vads_df,
    on=join_cols,
    how="inner",
)


# ============================================================
# Create arousal and valence categories
# 1,2,3 -> 0
# 4,5   -> 1
# ============================================================
text_vads_df["Arousal"] = pd.to_numeric(text_vads_df["Arousal"],errors="coerce")
text_vads_df["Valence"] = pd.to_numeric(text_vads_df["Valence"],errors="coerce")

text_vads_df = text_vads_df.dropna(subset=["Arousal","Valence"]).copy()

text_vads_df["Arousal"] = text_vads_df["Arousal"].astype(int)
text_vads_df["Valence"] = text_vads_df["Valence"].astype(int)

text_vads_df["arousal_category"] = text_vads_df["Arousal"].apply(
    lambda x: 0 if x in [1,2,3] else 1
)

text_vads_df["valence_category"] = text_vads_df["Valence"].apply(
    lambda x: 0 if x in [1,2,3] else 1
)


# ============================================================
# Create stimulus label from CMA
# LV -> 0
# HV -> 1
# ============================================================
text_vads_df["CMA"] = text_vads_df["CMA"].astype(str).str.strip()

text_vads_df["taskwiselabel"] = np.where(
    text_vads_df["CMA"].str.startswith("LV"),
    0,
    np.where(
        text_vads_df["CMA"].str.startswith("HV"),
        1,
        np.nan,
    )
)

# remove baseline / non LV-HV rows if any
text_vads_df = text_vads_df.dropna(subset=["taskwiselabel"]).copy()
text_vads_df["taskwiselabel"] = text_vads_df["taskwiselabel"].astype(int)


# ============================================================
# Store processed text + labels data
# ============================================================
output_path = Path("EEVR/Data_files/text_data_labels.csv")

text_vads_df.to_csv(
    output_path,
    index=False,
    encoding="utf-8"
)

print("Saved:", output_path)
print("Shape:", text_vads_df.shape)
display(text_vads_df.head())

Saved: EEVR/Data_files/text_data_labels.csv
Shape: (296, 13)


,Text Description,Participant ID,Playlist ID,Video ID,Segment Name,CMA,Valence,Arousal,Dominance,significance,arousal_category,valence_category,taskwiselabel
0,In this video childrens were shown whoes homes...,1,1,1,TheDisplaced,LVLA,4,3,2,3,0,1,0
1,In this video people and childrens were shown ...,1,1,2,HappyLand,LVLA,3,4,4,3,1,0,0
2,In this video a person who was confined in jai...,1,1,3,Jailbreak,LVHA,3,3,3,2,0,0,0
3,This video was about war and it impact all nat...,1,1,4,Warknownonation,LVHA,2,2,4,1,0,0,0
4,In this video I jumped into a canyon using a r...,1,1,5,CanyonSwing,HVHA,3,3,3,4,0,0,1


## Configuration


In [5]:
DATA_CONFIG = {
    "eda_path": "EEVR/Data_files/EDA_labels.csv",
    "ppg_path": "EEVR/Data_files/PPG_labels.csv",
    "text_path": "EEVR/Data_files/text_data_labels.csv",
}
identifiers = [
    "Participant ID",
    "Video ID",
    "arousal_category",
    "valence_category",
    "taskwiselabel",
]

In [6]:
# ============================================================
# Examples where valence_category and stimulus-label match / do not match
# ============================================================
cols_to_show = [
    "Participant ID",
    "Video ID",
    "CMA",
    "Arousal",
    "Valence",
    "arousal_category",
    "valence_category",
    "taskwiselabel",
]

matched_examples = (
    text_vads_df[
        text_vads_df["valence_category"].eq(text_vads_df["taskwiselabel"])
    ]
    [cols_to_show]
    .drop_duplicates()
    .head(10)
)

not_matched_examples = (
    text_vads_df[
        ~text_vads_df["valence_category"].eq(text_vads_df["taskwiselabel"])
    ]
    [cols_to_show]
    .drop_duplicates()
    .head(10)
)

# ============================================================
# Concatenate matched and not matched examples
# then sort by Participant ID and Video ID
# ============================================================
examples_latex_df = pd.concat(
    [
        matched_examples,
        not_matched_examples,
    ],
    ignore_index=True,
)

examples_latex_df = (
    examples_latex_df
    .sort_values(
        by=["Participant ID","Video ID"],
        ascending=[True,True]
    )
    .reset_index(drop=True)
)

display(examples_latex_df)


# ============================================================
# Store LaTeX table
# ============================================================
def latex_escape(x):
    if pd.isna(x):
        return ""

    x = str(x)

    replacements = {
        "\\": r"\textbackslash{}",
        "&": r"\&",
        "%": r"\%",
        "$": r"\$",
        "#": r"\#",
        "_": r"\_",
        "{": r"\{",
        "}": r"\}",
        "~": r"\textasciitilde{}",
        "^": r"\textasciicircum{}",
    }

    return "".join(replacements.get(ch,ch) for ch in x)


table_dir = Path("output/tables")
table_dir.mkdir(parents=True,exist_ok=True)

latex_path = table_dir / "valence_stimulus_match_examples.tex"

with open(latex_path,"w",encoding="utf-8") as f:
    f.write("\\begin{table}[htbp]\n")
    f.write("\\centering\n")
    f.write("\\scriptsize\n")
    f.write("\\caption{Examples comparing participant-reported valence category and stimulus label. Rows are sorted by participant ID and video ID.}\n")
    f.write("\\label{tab:valence_stimulus_match_examples}\n")
    f.write("\\begin{tabular}{lllccccc}\n")
    f.write("\\hline\n")
    f.write("\\rowcolor{gray!15}\n")
    f.write("\\textbf{Participant ID} & \\textbf{Video ID} & \\textbf{CMA} & \\textbf{Arousal} & \\textbf{Valence} & \\shortstack{\\textbf{Arousal}\\\\\\textbf{category}} & \\shortstack{\\textbf{Valence}\\\\\\textbf{category}} & \\shortstack{\\textbf{Stimulus}\\\\\\textbf{label}} \\\\\n")
    f.write("\\hline\n")

    for _,row in examples_latex_df.iterrows():
        values = [
            latex_escape(row["Participant ID"]),
            latex_escape(row["Video ID"]),
            latex_escape(row["CMA"]),
            latex_escape(row["Arousal"]),
            latex_escape(row["Valence"]),
            latex_escape(row["arousal_category"]),
            latex_escape(row["valence_category"]),
            latex_escape(row["taskwiselabel"]),
        ]

        f.write(" & ".join(values)+" \\\\\n")

    f.write("\\hline\n")
    f.write("\\end{tabular}\n")
    f.write("\\end{table}\n")

print("Saved LaTeX table:",latex_path)

,Participant ID,Video ID,CMA,Arousal,Valence,arousal_category,valence_category,taskwiselabel
0,1,1,LVLA,3,4,0,1,0
1,1,2,LVLA,4,3,1,0,0
2,1,3,LVHA,3,3,0,0,0
3,1,4,LVHA,2,2,0,0,0
4,1,5,HVHA,3,3,0,0,1
5,1,6,HVLA,3,3,0,0,1
6,1,7,HVHA,4,4,1,1,1
7,1,8,HVLA,2,4,0,1,1
8,2,1,LVLA,3,2,0,0,0
9,2,2,LVLA,3,1,0,0,0


Saved LaTeX table: output/tables/valence_stimulus_match_examples.tex


## Common utility functions


In [7]:
# ============================================================
# Utility: CSV read
# ============================================================
def csv_read(path):
    df = pd.read_csv(path)

    unnamed_cols = [
        col for col in df.columns
        if str(col).startswith("Unnamed")
    ]

    if len(unnamed_cols)>0:
        df = df.drop(columns=unnamed_cols)

    return df


# ============================================================
# Create Data ID column
# ============================================================
def create_dataid(df):
    df = df.copy()
    data_id = (
        1000 * df["Participant ID"].astype(int)
        + df["Video ID"].astype(int)
    )
    return data_id


# ============================================================
# Check if Data ID column exists
# ============================================================
def check_if_dataid_exists(df):
    if "Data ID" in df.columns:
        return True
    else:
        return False

In [8]:

#============================================================
# Read CSV files
# ============================================================
eda_df = clean_video_id(csv_read(DATA_CONFIG["eda_path"]))
ppg_df = clean_video_id(csv_read(DATA_CONFIG["ppg_path"]))
text_df = text_vads_df
# ============================================================
# Create Data ID
# ============================================================
if not check_if_dataid_exists(eda_df):    
    eda_df.insert(0, "Data ID", create_dataid(eda_df))
if not check_if_dataid_exists(ppg_df):
    ppg_df.insert(0, "Data ID", create_dataid(ppg_df))
if not check_if_dataid_exists(text_df):    
    text_df.insert(0, "Data ID", create_dataid(text_df))

# ============================================================
# Sort by Data ID to align rows
# ============================================================
eda_df = eda_df.sort_values("Data ID").reset_index(drop=True)
ppg_df = ppg_df.sort_values("Data ID").reset_index(drop=True)
text_df = text_df.sort_values("Data ID").reset_index(drop=True)

display(eda_df.head())
display(ppg_df.head())
display(text_df.head())

eda_df.to_csv(DATA_CONFIG["eda_path"], index=False)
ppg_df.to_csv(DATA_CONFIG["ppg_path"], index=False)
text_df.to_csv(DATA_CONFIG["text_path"], index=False)

,Data ID,ku_eda,sk_eda,dynrange,slope,variance,entropy,insc,fd_mean,max_scr,...,CMA,Video_ID_number,Valence,Arousal,Dominance,significance,arousal_category,valence_category,taskwiselabel,three_class_label
0,1000,0.044875,0.368040,1.0,0.434032,0.074698,0.648561,0.732354,0.479741,0.093830,...,Baseline,0,3.0,2.0,3.0,3.0,0,0,0,1
1,1001,0.253884,0.563838,1.0,0.469250,0.023473,0.680744,0.633802,0.508846,0.117726,...,LVLA,1,4.0,3.0,2.0,3.0,0,1,0,1
2,1002,0.041321,0.428991,1.0,0.411165,0.035743,0.475497,0.405047,0.426486,0.068658,...,LVLA,2,3.0,4.0,4.0,3.0,1,0,0,1
3,1003,0.054428,0.368401,1.0,0.438547,0.041062,0.638432,0.513979,0.495204,0.143321,...,LVHA,3,3.0,3.0,3.0,2.0,0,0,0,0
4,1004,0.119185,0.483140,1.0,0.410752,0.055979,0.633048,0.472319,0.423282,0.050042,...,LVHA,4,2.0,2.0,4.0,1.0,0,0,0,0


,Data ID,BPM,IBI,PPG_Rate_Mean,HRV_MedianNN,HRV_Prc20NN,HRV_MinNN,HRV_HTI,HRV_TINN,HRV_LF,...,Valence,Arousal,Dominance,significance,arousal_category,valence_category,CMA_numeric,task_valence,taskwiselabel,three_class_label
0,1000,0.462631,0.489583,0.762978,0.173913,0.256351,0.526316,0.240993,0.133929,0.323879,...,3.0,2.0,3.0,3.0,0,0,0,0,0,1
1,1001,0.391527,0.531250,0.787840,0.158103,0.265589,0.736842,0.114236,0.116071,0.254274,...,4.0,3.0,2.0,3.0,0,1,0,1,0,1
2,1002,0.482281,0.479167,0.748529,0.189723,0.297921,0.684211,0.053987,0.098214,0.744514,...,3.0,4.0,4.0,3.0,1,0,0,1,0,1
3,1003,0.399818,0.526042,0.779456,0.166008,0.277136,0.684211,0.118131,0.169643,0.296427,...,3.0,3.0,3.0,2.0,0,0,1,1,0,0
4,1004,0.416856,0.515625,0.791550,0.166008,0.277136,0.754386,0.119790,0.089286,0.881908,...,2.0,2.0,4.0,1.0,0,0,1,1,0,0


,Data ID,Text Description,Participant ID,Playlist ID,Video ID,Segment Name,CMA,Valence,Arousal,Dominance,significance,arousal_category,valence_category,taskwiselabel
0,1001,In this video childrens were shown whoes homes...,1,1,1,TheDisplaced,LVLA,4,3,2,3,0,1,0
1,1002,In this video people and childrens were shown ...,1,1,2,HappyLand,LVLA,3,4,4,3,1,0,0
2,1003,In this video a person who was confined in jai...,1,1,3,Jailbreak,LVHA,3,3,3,2,0,0,0
3,1004,This video was about war and it impact all nat...,1,1,4,Warknownonation,LVHA,2,2,4,1,0,0,0
4,1005,In this video I jumped into a canyon using a r...,1,1,5,CanyonSwing,HVHA,3,3,3,4,0,0,1


## EDA preprocessing


In [9]:
eda_features = [
    "ku_eda", "sk_eda", "dynrange", "slope", "variance",
    "entropy", "insc", "fd_mean", "max_scr", "min_scr",
    "nSCR", "meanAmpSCR", "meanRespSCR", "sumAmpSCR", "sumRespSCR",
]
eda_df = csv_read(DATA_CONFIG["eda_path"])

#### 4.1 Check and remove missing rows

In [10]:
# Check rows with any missing value
missing_rows = eda_df[eda_df.isna().any(axis=1)]
print("Number of rows with missing values:", len(missing_rows))

# Remove rows that contain at least one missing value
eda_df_clean = eda_df.dropna()

print("Original shape:", eda_df.shape)
print("Cleaned shape:", eda_df_clean.shape)

Number of rows with missing values: 0
Original shape: (333, 29)
Cleaned shape: (333, 29)


#### 4.2 Remove rows with Video ID 0 because corresponding text descriptions are not available

In [11]:
print("Before:", eda_df.shape) # baseline removed 
eda_df = eda_df[eda_df["Video ID"] != 0].reset_index(drop=True)
print("After:", eda_df.shape)

Before: (333, 29)
After: (296, 29)


#### 4.3 Standard normalization of EDA features

In [12]:
scaler = StandardScaler() # subtract feature column mean divided by feature column std
eda_df_scaled = eda_df.copy()
eda_df_scaled[eda_features] = scaler.fit_transform(eda_df_scaled[eda_features])

#### 4.4 Store preprocessed file in CSV format

In [13]:
eda_df_scaled.to_csv("EEVR/Data_files/EDA_preprocessed_aligned.csv", index=False)

## PPG preprocessing


In [14]:
ppg_features = [
    'BPM', 'IBI', 'PPG_Rate_Mean', 'HRV_MedianNN', 'HRV_Prc20NN',
    'HRV_MinNN', 'HRV_HTI', 'HRV_TINN', 'HRV_LF', 'HRV_VHF',
    'HRV_LFn', 'HRV_HFn', 'HRV_LnHF', 'HRV_SD1SD2', 'HRV_CVI',
    'HRV_PSS', 'HRV_PAS', 'HRV_PI', 'HRV_C1d', 'HRV_C1a',
    'HRV_DFA_alpha1', 'HRV_MFDFA_alpha1_Width', 'HRV_MFDFA_alpha1_Peak', 'HRV_MFDFA_alpha1_Mean', 'HRV_MFDFA_alpha1_Max',
    'HRV_MFDFA_alpha1_Delta', 'HRV_MFDFA_alpha1_Asymmetry', 'HRV_ApEn', 'HRV_ShanEn', 'HRV_FuzzyEn',
    'HRV_MSEn', 'HRV_CMSEn', 'HRV_RCMSEn', 'HRV_CD', 'HRV_HFD',
    'HRV_KFD', 'HRV_LZC',
]


ppg_df = csv_read(DATA_CONFIG["ppg_path"])

#### 4.1 Check and remove missing rows

In [15]:
# Check rows with any missing value
missing_rows = ppg_df[ppg_df.isna().any(axis=1)]

print("Number of rows with missing values:", len(missing_rows))

# Remove rows that contain at least one missing value
print("Original shape:", ppg_df.shape)
ppg_df = ppg_df.dropna().reset_index(drop=True)
print("Cleaned shape:", ppg_df.shape)

Number of rows with missing values: 0
Original shape: (333, 53)
Cleaned shape: (333, 53)


#### 4.2 Remove rows with Video ID 0 because corresponding text descriptions are not available

In [16]:
print("Before:", ppg_df.shape)

ppg_df = ppg_df[ppg_df["Video ID"] != 0].reset_index(drop=True)

print("After:", ppg_df.shape)

Before: (333, 53)
After: (296, 53)


#### 4.3 Standard normalization of PPG features

In [17]:

scaler = StandardScaler()
ppg_df_scaled = ppg_df.copy()
ppg_df_scaled[ppg_features] = scaler.fit_transform(ppg_df_scaled[ppg_features])

#### 4.4 Store preprocessed file in CSV format

In [18]:
ppg_df_scaled.to_csv("EEVR/Data_files/PPG_preprocessed_aligned.csv", index=False)

## TEXT preprocessing


### Text-specific utility functions


In [19]:


# ============================================================
# Initialize NLTK tools
# ============================================================
stop_words = set(stopwords.words("english"))
lemmatizer = WordNetLemmatizer()


# ============================================================
# Utility: Clean one text response
# ============================================================
def clean_text(text):
    """
    Clean one interview response using the NLTK pipeline:
    1. Replace missing values with empty strings
    2. Convert text to lowercase
    3. Tokenize text into words/tokens
    4. Remove punctuation and stop words
    5. Lemmatize words to their base form
    """

    if pd.isna(text):
        text = ""

    text = str(text).lower()
    tokens = word_tokenize(text)

    clean_tokens = []

    for token in tokens:
        # Remove punctuation characters from tokens
        token = token.translate(str.maketrans("", "", string.punctuation))
        token = token.strip()

        if token == "":
            continue

        if token in stop_words:
            continue

        token = lemmatizer.lemmatize(token)
        clean_tokens.append(token)

    return " ".join(clean_tokens)

In [20]:
display(text_df.head(5))

,Data ID,Text Description,Participant ID,Playlist ID,Video ID,Segment Name,CMA,Valence,Arousal,Dominance,significance,arousal_category,valence_category,taskwiselabel
0,1001,In this video childrens were shown whoes homes...,1,1,1,TheDisplaced,LVLA,4,3,2,3,0,1,0
1,1002,In this video people and childrens were shown ...,1,1,2,HappyLand,LVLA,3,4,4,3,1,0,0
2,1003,In this video a person who was confined in jai...,1,1,3,Jailbreak,LVHA,3,3,3,2,0,0,0
3,1004,This video was about war and it impact all nat...,1,1,4,Warknownonation,LVHA,2,2,4,1,0,0,0
4,1005,In this video I jumped into a canyon using a r...,1,1,5,CanyonSwing,HVHA,3,3,3,4,0,0,1


#### 4.2 Check missing values


In [21]:
# Check rows with any missing value
missing_rows = text_df[text_df.isna().any(axis=1)]

print("Number of rows with missing values:", len(missing_rows))

# Remove rows that contain at least one missing value
text_df_clean = text_df.dropna()

print("Original shape:", text_df.shape)
print("Cleaned shape:", text_df_clean.shape)

Number of rows with missing values: 0
Original shape: (296, 14)
Cleaned shape: (296, 14)


#### 4.3 Clean text descriptions using NLTK


In [22]:
cleaned_text = text_df_clean["Text Description"].apply(clean_text)
text_df_clean.insert(2, "Text", cleaned_text)
display(text_df_clean.head(5))

,Data ID,Text Description,Text,Participant ID,Playlist ID,Video ID,Segment Name,CMA,Valence,Arousal,Dominance,significance,arousal_category,valence_category,taskwiselabel
0,1001,In this video childrens were shown whoes homes...,video childrens shown whoes home destroyed war...,1,1,1,TheDisplaced,LVLA,4,3,2,3,0,1,0
1,1002,In this video people and childrens were shown ...,video people childrens shown slum area video e...,1,1,2,HappyLand,LVLA,3,4,4,3,1,0,0
2,1003,In this video a person who was confined in jai...,video person confined jail run aways jail nt f...,1,1,3,Jailbreak,LVHA,3,3,3,2,0,0,0
3,1004,This video was about war and it impact all nat...,video war impact nation equally video scene wa...,1,1,4,Warknownonation,LVHA,2,2,4,1,0,0,0
4,1005,In this video I jumped into a canyon using a r...,video jumped canyon using rope felt little sca...,1,1,5,CanyonSwing,HVHA,3,3,3,4,0,0,1


#### 4.4 Store preprocessed text file in CSV format


In [23]:
text_df_clean.to_csv("EEVR/Data_files/text_data_labels.csv", index=False)
eda_df.to_csv("EEVR/Data_files/EDA_preprocessed_aligned.csv", index=False)
ppg_df.to_csv("EEVR/Data_files/PPG_preprocessed_aligned.csv", index=False)
text_df_clean.to_csv("EEVR/Data_files/TEXT_preprocessed_aligned.csv", index=False)